## m steering

In [8]:
import glob
import numpy as np
from scipy.linalg import fractional_matrix_power
import time

def read_vectors_glob(pattern: str) -> np.array:
    results = []
    for path in glob.iglob(pattern):
        results.extend(np.load(path, allow_pickle=True))
    return results

In [20]:
neg_vectors = read_vectors_glob('../hidden_states/pos_vectors_horse_*.npy')
pos_vectors = read_vectors_glob('../hidden_states/pos_vectors_motorcycle_*.npy')

In [21]:
result = {}
for denoising_step in pos_vectors[0].keys():
    result[denoising_step] = {}
    for block in pos_vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(pos_vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            pos = np.stack([vector[denoising_step][block][layer] for vector in pos_vectors]).astype(np.float32)
            neg = np.stack([vector[denoising_step][block][layer] for vector in neg_vectors]).astype(np.float32)
            steering_vector = np.mean(pos, axis=0) - np.mean(neg, axis=0)
            steering_vector /= np.linalg.norm(steering_vector)
            result[denoising_step][block].append(steering_vector)

Processing step=0, block=down, layer=0
Processing step=0, block=down, layer=1
Processing step=0, block=down, layer=2
Processing step=0, block=down, layer=3
Processing step=0, block=down, layer=4
Processing step=0, block=down, layer=5
Processing step=0, block=down, layer=6
Processing step=0, block=down, layer=7
Processing step=0, block=down, layer=8
Processing step=0, block=down, layer=9
Processing step=0, block=down, layer=10
Processing step=0, block=down, layer=11
Processing step=0, block=down, layer=12
Processing step=0, block=down, layer=13
Processing step=0, block=down, layer=14
Processing step=0, block=down, layer=15
Processing step=0, block=down, layer=16
Processing step=0, block=down, layer=17
Processing step=0, block=down, layer=18
Processing step=0, block=down, layer=19
Processing step=0, block=down, layer=20
Processing step=0, block=down, layer=21
Processing step=0, block=down, layer=22
Processing step=0, block=down, layer=23
Processing step=0, block=up, layer=0
Processing st

In [22]:
import pickle

with open('../steering_vectors/horse_to_motorcycle_laion_steering_vectors.pickle', 'wb') as fout:
    pickle.dump(result, fout)

## mm steering

In [15]:
result = {}
for denoising_step in pos_vectors[0].keys():
    result[denoising_step] = {}
    for block in pos_vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(pos_vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            start = time.time()
            pos = np.stack([vector[denoising_step][block][layer] for vector in pos_vectors]).astype(np.float64)
            pos /= np.linalg.norm(pos, axis=1, keepdims=True)

            neg = np.stack([vector[denoising_step][block][layer] for vector in neg_vectors]).astype(np.float64)
            neg /= np.linalg.norm(neg, axis=1, keepdims=True)

            mu_pos = np.mean(pos, axis=0)
            sigma_pos = np.dot(pos.T, pos) / (pos.shape[0] - 1)
            sigma_pos -= np.outer(mu_pos, mu_pos)

            mu_neg = np.mean(neg, axis=0)
            sigma_neg = np.dot(neg.T, neg) / (neg.shape[0] - 1)
            sigma_neg -= np.outer(mu_neg, mu_neg)

            
            sigma_neg_half = fractional_matrix_power(sigma_neg, 0.5)
            sigma_neg_minus_half = fractional_matrix_power(sigma_neg, -0.5)
            W = fractional_matrix_power(sigma_neg_half @ sigma_pos @ sigma_neg_half, 0.5)
            W = sigma_neg_minus_half @ W @ sigma_neg_minus_half

            b = - W @ mu_neg + mu_pos

            print(f'Took: {time.time() - start:.2f} s')

            if W.dtype == np.complex128:
                print(f'Got unexpected complex values for step={denoising_step}, block={block}, layer={layer}, truncating...')
                W = np.real(W)
                b = np.real(b)
            
            result[denoising_step][block].append((W.astype(np.float32), b.astype(np.float32)))

Processing step=0, block=down, layer=0
Took: 0.87 s
Processing step=0, block=down, layer=1
Took: 0.84 s
Processing step=0, block=down, layer=2
Took: 0.84 s
Processing step=0, block=down, layer=3
Took: 0.85 s
Processing step=0, block=down, layer=4
Took: 3.82 s
Processing step=0, block=down, layer=5
Took: 3.60 s
Processing step=0, block=down, layer=6
Took: 5.82 s
Got unexpected complex values for step=0, block=down, layer=6, truncating...
Processing step=0, block=down, layer=7
Took: 5.87 s
Got unexpected complex values for step=0, block=down, layer=7, truncating...
Processing step=0, block=down, layer=8
Took: 3.64 s
Processing step=0, block=down, layer=9
Took: 3.64 s
Processing step=0, block=down, layer=10
Took: 3.60 s
Processing step=0, block=down, layer=11
Took: 3.71 s
Processing step=0, block=down, layer=12
Took: 3.70 s
Processing step=0, block=down, layer=13
Took: 3.71 s
Processing step=0, block=down, layer=14
Took: 5.81 s
Got unexpected complex values for step=0, block=down, layer=1

In [16]:
import pickle

with open('../steering_vectors/horse_to_motorcycle_laion_mm_steering_vectors.pickle', 'wb') as fout:
    pickle.dump(result, fout)

In [17]:
pos_vectors, neg_vectors = neg_vectors, pos_vectors

In [18]:
result = {}
for denoising_step in pos_vectors[0].keys():
    result[denoising_step] = {}
    for block in pos_vectors[0][denoising_step].keys():
        result[denoising_step][block] = []
        for layer in range(len(pos_vectors[0][denoising_step][block])):
            print(f'Processing step={denoising_step}, block={block}, layer={layer}')
            start = time.time()
            pos = np.stack([vector[denoising_step][block][layer] for vector in pos_vectors]).astype(np.float64)
            pos /= np.linalg.norm(pos, axis=1, keepdims=True)

            neg = np.stack([vector[denoising_step][block][layer] for vector in neg_vectors]).astype(np.float64)
            neg /= np.linalg.norm(neg, axis=1, keepdims=True)

            mu_pos = np.mean(pos, axis=0)
            sigma_pos = np.dot(pos.T, pos) / (pos.shape[0] - 1)
            sigma_pos -= np.outer(mu_pos, mu_pos)

            mu_neg = np.mean(neg, axis=0)
            sigma_neg = np.dot(neg.T, neg) / (neg.shape[0] - 1)
            sigma_neg -= np.outer(mu_neg, mu_neg)

            
            sigma_neg_half = fractional_matrix_power(sigma_neg, 0.5)
            sigma_neg_minus_half = fractional_matrix_power(sigma_neg, -0.5)
            W = fractional_matrix_power(sigma_neg_half @ sigma_pos @ sigma_neg_half, 0.5)
            W = sigma_neg_minus_half @ W @ sigma_neg_minus_half

            b = - W @ mu_neg + mu_pos

            print(f'Took: {time.time() - start:.2f} s')

            if W.dtype == np.complex128:
                print(f'Got unexpected complex values for step={denoising_step}, block={block}, layer={layer}, truncating...')
                W = np.real(W)
                b = np.real(b)
            
            result[denoising_step][block].append((W.astype(np.float32), b.astype(np.float32)))

Processing step=0, block=down, layer=0
Took: 1.07 s
Processing step=0, block=down, layer=1
Took: 0.96 s
Processing step=0, block=down, layer=2
Took: 1.05 s
Processing step=0, block=down, layer=3
Took: 0.97 s
Processing step=0, block=down, layer=4
Took: 4.20 s
Processing step=0, block=down, layer=5
Took: 9.43 s
Got unexpected complex values for step=0, block=down, layer=5, truncating...
Processing step=0, block=down, layer=6
Took: 8.02 s
Got unexpected complex values for step=0, block=down, layer=6, truncating...
Processing step=0, block=down, layer=7
Took: 6.25 s
Got unexpected complex values for step=0, block=down, layer=7, truncating...
Processing step=0, block=down, layer=8
Took: 3.99 s
Processing step=0, block=down, layer=9
Took: 3.77 s
Processing step=0, block=down, layer=10
Took: 3.90 s
Processing step=0, block=down, layer=11
Took: 3.69 s
Processing step=0, block=down, layer=12
Took: 3.78 s
Processing step=0, block=down, layer=13
Took: 3.68 s
Processing step=0, block=down, layer=

In [19]:
import pickle

with open('../steering_vectors/motorcycle_to_horse_laion_mm_inverse_steering_vectors.pickle', 'wb') as fout:
    pickle.dump(result, fout)

In [35]:
import pickle

with open('CASteer/mickey_laion_mm_steering_vectors.pickle', 'rb') as fin:
    result = pickle.load(fin)

In [74]:
from scipy.linalg import fractional_matrix_power

In [75]:
W, b = result[0]['down'][12]

In [76]:
W

array([[ 9.87304621e-01,  4.30656828e-02,  1.21133825e-02, ...,
        -1.66107101e-03,  6.09483959e-03,  3.06179327e-04],
       [-3.26536440e-03,  9.75993828e-01,  7.95705294e-03, ...,
         2.53791023e-03, -6.68410038e-03,  3.41204331e-03],
       [-3.72342627e-03,  5.24025991e-03,  9.59643629e-01, ...,
        -2.48833296e-03,  3.90967577e-03,  1.84640455e-03],
       ...,
       [-1.27566990e-03, -1.13116318e-02,  2.61789753e-02, ...,
         9.91237713e-01, -1.34029297e-03, -1.36469132e-03],
       [-1.04599299e-03, -4.45986643e-02,  1.93311209e-02, ...,
         2.88671404e-03,  9.60568226e-01, -3.77163922e-03],
       [-2.04978859e-03,  3.39126281e-02, -6.76758640e-03, ...,
         1.92174849e-03,  4.16719427e-03,  9.91486981e-01]],
      shape=(1280, 1280))

In [80]:
fractional_matrix_power(W, 0.5)

array([[ 9.87554507e-01+7.60988823e-18j,  1.75050245e-02-7.16830086e-18j,
         6.83772205e-03+4.30149788e-17j, ...,
        -1.28758291e-04+2.62742268e-18j,  2.14492409e-03+4.15611667e-18j,
        -2.45413685e-04-2.00404873e-17j],
       [-1.41406079e-03+1.52368785e-17j,  9.84064159e-01-9.09395088e-18j,
         5.33487240e-03-1.37801324e-17j, ...,
         1.22598661e-03+1.21790772e-17j, -3.56832047e-03-2.22887668e-17j,
         1.61634808e-03+1.19325793e-17j],
       [-2.27727207e-03+1.71066838e-17j, -6.33509690e-04+1.31701880e-17j,
         9.72770690e-01+4.56759230e-17j, ...,
        -1.17237758e-03-4.24536938e-18j,  1.78990510e-03+3.26599789e-17j,
         1.38404062e-04+1.65839119e-17j],
       ...,
       [ 3.59000876e-04+2.46553883e-17j,  1.89474955e-05-1.31034290e-17j,
         1.78680505e-02+8.80777486e-19j, ...,
         9.87679876e-01+1.66026898e-17j, -6.22709796e-04+4.38279064e-18j,
        -1.88015954e-04-1.41190771e-17j],
       [-4.23767490e-04+2.34663863e-17j, -1.

In [81]:
torch_fractional_matrix_power(torch.Tensor(W), 0.5)

tensor([[ 9.8756e-01-1.2887e-08j,  1.7505e-02+3.4576e-06j,
          6.8363e-03-1.2529e-06j,  ...,
         -1.2910e-04-3.6861e-07j,  2.1455e-03+1.3837e-06j,
         -2.4722e-04-4.0945e-07j],
        [-1.4095e-03+3.0404e-07j,  9.8407e-01-2.5525e-06j,
          5.3415e-03-6.6250e-07j,  ...,
          1.2252e-03-1.5090e-06j, -3.5721e-03-3.1923e-07j,
          1.6156e-03-3.1944e-07j],
        [-2.2757e-03+8.3566e-07j, -6.3747e-04-4.2088e-06j,
          9.7278e-01-9.3184e-07j,  ...,
         -1.1712e-03-5.1939e-07j,  1.7890e-03+2.4646e-06j,
          1.4132e-04+1.7531e-06j],
        ...,
        [ 3.5552e-04+6.8984e-07j,  1.9836e-05+1.6842e-06j,
          1.7868e-02+9.4772e-07j,  ...,
          9.8768e-01-1.2101e-07j, -6.2216e-04+3.2806e-07j,
         -1.8849e-04-7.3240e-08j],
        [-4.2571e-04-1.1361e-06j, -1.1499e-02+3.4615e-07j,
          1.8601e-02+3.0215e-06j,  ...,
          1.2400e-03+1.2065e-07j,  9.7608e-01+1.5653e-07j,
         -1.5779e-03+9.7121e-07j],
        [-1.0120e-03+3

In [63]:
(np.eye(W.shape[0]) - fractional_matrix_power(W, 1.5)) @ fractional_matrix_power(np.eye(W.shape[0]) - W, -1) @ b

array([ 0.01813325+5.51233013e-16j, -0.0029109 +7.96668669e-16j,
        0.01075266+1.32376058e-15j, ..., -0.0120704 -8.83834059e-16j,
       -0.00204257+2.43729800e-15j,  0.00534722+1.64256740e-15j],
      shape=(1280,))

In [64]:
b

array([ 0.00806543, -0.00187049,  0.00491921, ..., -0.00487827,
        0.00187789,  0.00037014], shape=(1280,))

In [91]:
def fractional_matrix_power(mat: torch.Tensor, alpha: float) -> torch.Tensor:
    device = mat.device
    if mat.device.type == 'mps':  # Workaround because MPS does not yet support torch.linalg.eig
        mat = mat.cpu()
    evals, evecs = torch.linalg.eig(mat)
    return (evecs @ torch.diag(evals ** alpha) @ evecs.inverse()).to(device)

In [92]:
fractional_matrix_power(torch.Tensor(W).to('mps'), 0.1)

tensor([[ 9.9661e-01+1.5825e-07j,  3.1239e-03+3.4664e-06j,
          1.7157e-03-1.1962e-06j,  ...,
          5.4650e-05-2.3014e-07j,  3.1233e-04+1.4054e-06j,
         -7.2402e-05-3.5203e-07j],
        [-2.4095e-04+5.4035e-08j,  9.9620e-01-2.2437e-06j,
          1.3829e-03-6.7741e-07j,  ...,
          2.2114e-04-1.6289e-06j, -7.6025e-04-4.4692e-07j,
          3.2111e-04-4.0846e-07j],
        [-5.1475e-04+1.0024e-06j, -6.2550e-04-4.1243e-06j,
          9.9332e-01-9.8481e-07j,  ...,
         -2.2301e-04-5.3816e-07j,  3.3986e-04+2.2665e-06j,
         -6.5179e-05+1.7997e-06j],
        ...,
        [ 1.9646e-04+4.5477e-07j,  9.2465e-04+1.7877e-06j,
          4.6155e-03+6.4888e-07j,  ...,
          9.9639e-01-2.6066e-07j, -1.3320e-04+4.6853e-07j,
          4.1826e-05+2.0373e-07j],
        [-6.0575e-05-1.2597e-06j, -6.2763e-04+4.6365e-07j,
          5.5065e-03+2.9990e-06j,  ...,
          1.9368e-04-2.1361e-07j,  9.9452e-01+2.1608e-08j,
         -2.6316e-04+7.9113e-07j],
        [-1.9453e-04+1

In [85]:
torch.Tensor(W).to('mps').device.type

'mps'

In [74]:
torch.tensor([1, 2, 3]).half()

tensor([1., 2., 3.], dtype=torch.float16)

## Re-LAION

In [1]:
from datasets import load_dataset

ds = load_dataset(
    "laion/relaion2B-en-research",
    cache_dir='./cache',
    data_files=[
        f'part-{i:05}-b31ba513-fc6b-4450-9ba4-a1bba183f408-c000.snappy.parquet'
            for i in range(1)
    ]   
)

/media/1TB_welcome/mmsteer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
1

1

In [3]:
df = ds['train'].to_pandas()

In [4]:
import re

def generate_concept(dataset: list[str], concept: str) -> tuple[list[str], list[str]]:
    pattern = re.compile(f'(^|[\\s.,-:;]){concept}($|[\\s.,-:;])', flags=re.IGNORECASE)
    pos, neg = [], []
    for text in dataset:
        if text is None:
            continue
        if pattern.search(text) is not None:
            pos.append(text)
        else:
            neg.append(text)
    return pos, neg

In [5]:
pos_sent, neg_sent = generate_concept(df['caption'].values, 'motorcycle|bike')

In [6]:
len(pos_sent)

59920

In [33]:
len(neg_sent)

16894144

In [8]:
with open('CASteer/concept_prompts/motorcycle_pos_sentence.txt', 'w') as fout:
    for s in pos_sent[:30000]:
        print(s, file=fout)

In [30]:
with open('CASteer/concept_prompts/nudity_neg_sentence.txt', 'w') as fout:
    for s in neg_sent[:30000]:
        print(s, file=fout)

In [57]:
with open('CASteer/pos_sentence.txt', 'r') as fin:
    prompts_pos = list(map(str.strip, fin.readlines()))

In [93]:
1

1

### Prompts for metrics

In [94]:
imagenet_templates = [
    'a bad photo of a {}.',
    'a photo of many {}.',
    'a sculpture of a {}.',
    'a photo of the hard to see {}.',
    'a low resolution photo of the {}.',
    'a rendering of a {}.',
    'graffiti of a {}.',
    'a bad photo of the {}.',
    'a cropped photo of the {}.',
    'a tattoo of a {}.',
    'the embroidered {}.',
    'a photo of a hard to see {}.',
    'a bright photo of a {}.',
    'a photo of a clean {}.',
    'a photo of a dirty {}.',
    'a dark photo of the {}.',
    'a drawing of a {}.',
    'a photo of my {}.',
    'the plastic {}.',
    'a photo of the cool {}.',
    'a close-up photo of a {}.',
    'a black and white photo of the {}.',
    'a painting of the {}.',
    'a painting of a {}.',
    'a pixelated photo of the {}.',
    'a sculpture of the {}.',
    'a bright photo of the {}.',
    'a cropped photo of a {}.',
    'a plastic {}.',
    'a photo of the dirty {}.',
    'a jpeg corrupted photo of a {}.',
    'a blurry photo of the {}.',
    'a photo of the {}.',
    'a good photo of the {}.',
    'a rendering of the {}.',
    'a {} in a video game.',
    'a photo of one {}.',
    'a doodle of a {}.',
    'a close-up photo of the {}.',
    'a photo of a {}.',
    'the origami {}.',
    'the {} in a video game.',
    'a sketch of a {}.',
    'a doodle of the {}.',
    'a origami {}.',
    'a low resolution photo of a {}.',
    'the toy {}.',
    'a rendition of the {}.',
    'a photo of the clean {}.',
    'a photo of a large {}.',
    'a rendition of a {}.',
    'a photo of a nice {}.',
    'a photo of a weird {}.',
    'a blurry photo of a {}.',
    'a cartoon {}.',
    'art of a {}.',
    'a sketch of the {}.',
    'a embroidered {}.',
    'a pixelated photo of a {}.',
    'itap of the {}.',
    'a jpeg corrupted photo of the {}.',
    'a good photo of a {}.',
    'a plushie {}.',
    'a photo of the nice {}.',
    'a photo of the small {}.',
    'a photo of the weird {}.',
    'the cartoon {}.',
    'art of the {}.',
    'a drawing of the {}.',
    'a photo of the large {}.',
    'a black and white photo of a {}.',
    'the plushie {}.',
    'a dark photo of a {}.',
    'itap of a {}.',
    'graffiti of the {}.',
    'a toy {}.',
    'itap of my {}.',
    'a photo of a cool {}.',
    'a photo of a small {}.',
    'a tattoo of the {}.',
]

In [98]:
mickey_prompts = [s.format('Snoopy') for s in imagenet_templates]

In [99]:
for s in mickey_prompts:
    print(repr(s))

'a bad photo of a Snoopy.'
'a photo of many Snoopy.'
'a sculpture of a Snoopy.'
'a photo of the hard to see Snoopy.'
'a low resolution photo of the Snoopy.'
'a rendering of a Snoopy.'
'graffiti of a Snoopy.'
'a bad photo of the Snoopy.'
'a cropped photo of the Snoopy.'
'a tattoo of a Snoopy.'
'the embroidered Snoopy.'
'a photo of a hard to see Snoopy.'
'a bright photo of a Snoopy.'
'a photo of a clean Snoopy.'
'a photo of a dirty Snoopy.'
'a dark photo of the Snoopy.'
'a drawing of a Snoopy.'
'a photo of my Snoopy.'
'the plastic Snoopy.'
'a photo of the cool Snoopy.'
'a close-up photo of a Snoopy.'
'a black and white photo of the Snoopy.'
'a painting of the Snoopy.'
'a painting of a Snoopy.'
'a pixelated photo of the Snoopy.'
'a sculpture of the Snoopy.'
'a bright photo of the Snoopy.'
'a cropped photo of a Snoopy.'
'a plastic Snoopy.'
'a photo of the dirty Snoopy.'
'a jpeg corrupted photo of a Snoopy.'
'a blurry photo of the Snoopy.'
'a photo of the Snoopy.'
'a good photo of the Snoop